# Cross-Sectional Stock Return Prediction

This notebook runs the attention experiment from `run_project_mhattn.py`.
This multi-seed copy averages metrics across several random seeds and uses representative runs for plots.


In [ ]:
import importlib
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython import get_ipython

ip = get_ipython()
if ip is not None:
    ip.run_line_magic("matplotlib", "inline")
    try:
        from matplotlib_inline.backend_inline import set_matplotlib_formats
        set_matplotlib_formats("png")
    except Exception:
        pass

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import stock_return_core as srp
import run_project_mhattn as mh

srp = importlib.reload(srp)
mh = importlib.reload(mh)
print(srp.__file__)
print(mh.__file__)


In [ ]:
# Quick run defaults for the averaged attention experiment.
START_DATE = "2015-01-01"
END_DATE = "2025-01-01"
LOOKBACK = 60
HORIZON = 5
UNIVERSE = "small"  # small | sp500 | nasdaq100 | auto
MODELS = ["RNN", "LSTM", "GRU", "TRANSFORMER"]
ATTN_HEADS = 4
DEVICE = mh.resolve_device()
SEEDS = [40, 41, 42]

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True

UNIVERSE_TO_TICKERS = {
    "small": srp.DEFAULT_LIQUID_TICKERS,
    "sp500": srp.SP500_TICKERS,
    "nasdaq100": srp.NASDAQ100_TICKERS,
    "auto": None,
}
TICKERS = UNIVERSE_TO_TICKERS[UNIVERSE]
print(f"Universe: {UNIVERSE}")
print(f"Seeds: {SEEDS}")

# Universe filter: keep stocks that cover nearly the full sample window.
START_BUFFER_DAYS = 5
END_BUFFER_DAYS = 5
MIN_COVERAGE_RATIO = 0.95


In [ ]:
raw_prices = srp.download_price_history(start=START_DATE, end=END_DATE, tickers=TICKERS)

flat_prices = srp.flatten_price_frame(raw_prices).dropna(subset=["Close"]).copy()
all_dates = pd.Index(sorted(pd.to_datetime(flat_prices["Date"].unique())))
date_to_pos = {date: idx for idx, date in enumerate(all_dates)}

coverage = (
    flat_prices.groupby("Ticker")
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        n_dates=("Date", "nunique"),
    )
    .reset_index()
)
coverage["first_pos"] = pd.to_datetime(coverage["first_date"]).map(date_to_pos)
coverage["last_pos"] = pd.to_datetime(coverage["last_date"]).map(date_to_pos)
coverage["coverage_ratio"] = coverage["n_dates"] / len(all_dates)

eligible_tickers = coverage.loc[
    (coverage["first_pos"] <= START_BUFFER_DAYS)
    & (coverage["last_pos"] >= len(all_dates) - 1 - END_BUFFER_DAYS)
    & (coverage["coverage_ratio"] >= MIN_COVERAGE_RATIO),
    "Ticker",
].tolist()

filtered_prices = raw_prices.loc[
    :,
    raw_prices.columns.get_level_values(0).isin(eligible_tickers),
]

print(
    f"Kept {len(eligible_tickers)} / {coverage.shape[0]} tickers "
    f"with >= {MIN_COVERAGE_RATIO:.0%} coverage and near-full sample history."
)
print(
    "Dropped examples:",
    coverage.loc[~coverage["Ticker"].isin(eligible_tickers), "Ticker"].head(10).tolist(),
)

experiment_data = srp.prepare_experiment_data(
    filtered_prices,
    horizon=HORIZON,
    lookback=LOOKBACK,
    train_size=0.7,
    val_size=0.15,
)
for split_name, split_df in experiment_data.splits.items():
    print(split_name, split_df["Date"].min(), split_df["Date"].max(), len(split_df))


In [ ]:
MODEL_CONFIGS = {
    "RNN": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=1,
        dropout=0.0,
        learning_rate=1e-4,
        weight_decay=1e-5,
        max_epochs=20,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "LSTM": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=2,
        dropout=0.15,
        learning_rate=3e-4,
        weight_decay=1e-5,
        max_epochs=25,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "GRU": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=2,
        dropout=0.15,
        learning_rate=1e-4,
        weight_decay=1e-5,
        max_epochs=25,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "TRANSFORMER": srp.TrainConfig(
        batch_size=256,
        hidden_dim=160,
        num_layers=3,
        dropout=0.15,
        num_heads=8,
        learning_rate=3e-4,
        weight_decay=5e-5,
        max_epochs=35,
        patience=10,
        grad_clip=1.0,
        device=DEVICE,
    ),
}

input_dim = len(experiment_data.feature_columns)
results_by_seed = {}
seed_summaries = []

for seed in SEEDS:
    print(f"=== Seed {seed} ===")
    seed_results = {}
    for model_name in MODELS:
        train_config = MODEL_CONFIGS[model_name]
        set_seed(seed)
        print(f"=== Notebook model: {model_name} (seed={seed}) ===")
        if model_name == "RNN":
            model = srp.build_model(
                model_name="RNN",
                input_dim=input_dim,
                hidden_dim=train_config.hidden_dim,
                num_layers=train_config.num_layers,
                dropout=train_config.dropout,
            )
        elif model_name in {"LSTM", "GRU"}:
            model = mh.AttentionSequenceRegressor(
                input_dim=input_dim,
                hidden_dim=train_config.hidden_dim,
                num_layers=train_config.num_layers,
                dropout=train_config.dropout,
                cell_type=model_name,
                num_attn_heads=ATTN_HEADS,
            )
        elif model_name == "TRANSFORMER":
            model = srp.build_model(
                model_name="TRANSFORMER",
                input_dim=input_dim,
                hidden_dim=train_config.hidden_dim,
                num_layers=train_config.num_layers,
                dropout=train_config.dropout,
                num_heads=train_config.num_heads,
            )
        else:
            raise ValueError(f"Unsupported model: {model_name}")
        seed_results[model_name] = mh.run_experiment(
            model,
            model_name,
            experiment_data,
            train_config,
        )
    results_by_seed[seed] = seed_results
    seed_summary = srp.build_summary_frame(seed_results).copy()
    seed_summary["seed"] = seed
    seed_summaries.append(seed_summary)

all_seed_summaries = pd.concat(seed_summaries, ignore_index=True)
metric_cols = [col for col in all_seed_summaries.columns if col not in {"model", "seed"}]
summary = (
    all_seed_summaries.groupby("model", as_index=False)[metric_cols]
    .mean(numeric_only=True)
    .sort_values("mean_ic", ascending=False)
    .reset_index(drop=True)
)
summary_std = (
    all_seed_summaries.groupby("model", as_index=False)[metric_cols]
    .std(numeric_only=True)
    .rename(columns={col: f"{col}_seed_std" for col in metric_cols})
)
summary = summary.merge(summary_std, left_on="model", right_on="model", how="left")

representative_seed_by_model = {}
for model_name in MODELS:
    target_mean_ic = float(summary.loc[summary["model"] == model_name, "mean_ic"].iloc[0])
    model_rows = all_seed_summaries[all_seed_summaries["model"] == model_name].copy()
    representative_seed = int(model_rows.iloc[(model_rows["mean_ic"] - target_mean_ic).abs().argmin()]["seed"])
    representative_seed_by_model[model_name] = representative_seed

results = {
    model_name: results_by_seed[representative_seed_by_model[model_name]][model_name]
    for model_name in MODELS
}
print("Representative seeds:", representative_seed_by_model)


In [ ]:
display(summary)
display(all_seed_summaries)


# Result Visualizations

Plots for loss curves, IC/RankIC, portfolio performance, and model comparison.


In [ ]:
best_model = summary.iloc[0]["model"]
best_seed = representative_seed_by_model[best_model]
print(f"Visualizing model: {best_model} (representative seed={best_seed})")
srp.plot_training_histories(results)


In [ ]:
srp.plot_ic_series(results, best_model, split="test", rolling_window=20)


In [ ]:
import matplotlib.pyplot as plt

for model_name, result in results.items():
    if "portfolio_curve" not in result:
        portfolio_curve, portfolio_summary = srp.backtest_long_short(result["test_predictions"])
        result["portfolio_curve"] = portfolio_curve
        result["portfolio_summary"] = portfolio_summary

portfolio_fig = srp.plot_portfolio_curve(results, best_model)
summary_fig = srp.plot_summary_bars(summary)
display(portfolio_fig)
display(summary_fig)
plt.close(portfolio_fig)
plt.close(summary_fig)


# Prediction diagnostics

Inspect raw predictions, sign accuracy, and the prediction-vs-target scatter plot.


In [ ]:
model_name = best_model
seed_used = representative_seed_by_model[model_name]
print(f"Prediction diagnostics use representative seed: {seed_used}")
pred = results[model_name]["test_predictions"].copy()
pred.head()


In [ ]:
pred["pred_sign"] = (pred["prediction"] > 0).astype(int)
pred["true_sign"] = (pred["target"] > 0).astype(int)

sign_accuracy = (pred["pred_sign"] == pred["true_sign"]).mean()
daily_sign_accuracy = pred.groupby("Date").apply(
    lambda x: ((x["prediction"] > 0) == (x["target"] > 0)).mean(),
    include_groups=False,
)

print("Overall sign accuracy:", round(float(sign_accuracy), 4))
daily_sign_accuracy.describe()


In [ ]:
import matplotlib.pyplot as plt

latest_day = pred["Date"].max()
latest_slice = pred[pred["Date"] == latest_day].copy()
display(latest_slice.sort_values("prediction", ascending=False).head(10))

ax = latest_slice.plot.scatter(
    x="prediction",
    y="target",
    figsize=(6, 5),
    alpha=0.7,
    title=f"{model_name} prediction vs target on {latest_day.date()}"
)
ax.axhline(0.0, color="black", linewidth=1, alpha=0.6)
ax.axvline(0.0, color="black", linewidth=1, alpha=0.6)
plt.show()
